# 05 — SIR / SEIR Compartmental Model

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
import numpy as np

## 1. Load country data

In [2]:
featured = pd.read_csv('../data/processed/features.csv', parse_dates=['date'])

COUNTRY = 'United States'
POPULATION = 330_000_000

cdf = featured[featured.country == COUNTRY].sort_values('date')
I_obs = cdf['new_cases_7d'].fillna(0).values
print(f"Series length: {len(I_obs)} days")

Series length: 0 days


## 2. Fit SIR model

In [4]:
# from src.models.sir_model import SIRModel

# sir = SIRModel()
# sir.fit(I_obs, N=POPULATION)
# print(f"R₀ = {sir.r0:.3f}")
# sir.plot(I_obs, country=COUNTRY)
import numpy as np
from src.models.sir_model import SIRModel

# Example observed infections (replace with your real data)
# Ensure this is NOT empty
I_obs = np.array([10, 15, 20, 25, 30])  

POPULATION = 1_000_000   # replace with actual population size
COUNTRY = "India"        # or whichever country you’re modeling

# Initialize model
sir = SIRModel()

# Validate input before fitting
if I_obs is None or len(I_obs) == 0:
    raise ValueError("I_obs is empty. Please provide observed infection data.")
else:
    sir.fit(I_obs, N=POPULATION)
    print(f"R₀ = {sir.r0:.3f}")
    sir.plot(I_obs, country=COUNTRY)


  SIR fit  β=0.3856  γ=0.0984  R0=3.92
R₀ = 3.921
  SIR plot → outputs/plots/SIR_India.png


## 3. Fit SEIR model

In [5]:
from src.models.sir_model import SEIRModel

seir = SEIRModel(sigma=1/5)
seir.fit(I_obs, N=POPULATION)
print(f"SEIR R₀ = {seir.r0:.3f}")

  SEIR fit  β=0.5206  γ=0.0100  σ=0.2000  R0=52.06
SEIR R₀ = 52.059


## 4. 60-day forecast

In [6]:
import matplotlib.pyplot as plt

forecast = sir.predict_from(I_obs[-1], days=60)
fig, ax = plt.subplots(figsize=(12, 4))
fig.patch.set_facecolor('#050a0f')
ax.set_facecolor('#0a1520')
ax.plot(I_obs[-90:], color='#7a9e90', label='Observed (last 90d)')
ax.plot(range(90, 90+60), forecast, color='#00c896',
        linestyle='--', label='SIR 60-day forecast')
ax.set_title(f'{COUNTRY} — 60-day SIR Forecast  R₀={sir.r0:.2f}', color='#d0e8e0')
ax.legend(facecolor='#0d1b2a', edgecolor='#1a3040', labelcolor='#7a9e90')
ax.grid(True, color='#0f1e2e')
plt.tight_layout()
plt.show()

C:\Users\Dell\AppData\Local\Temp\ipykernel_3476\3134614075.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. R₀ map across countries

In [8]:
# results = []
# for country in featured['country'].unique()[:20]:
#     sub = featured[featured.country == country].sort_values('date')
#     I = sub['new_cases_7d'].fillna(0).values
#     pop = int(sub['population'].median()) if 'population' in sub else 1_000_000
#     if len(I) < 50 or I.max() < 10:
#         continue
#     try:
#         s = SIRModel()
#         s.fit(I, N=max(pop, 1_000_000))
#         results.append({'country': country, 'R0': s.r0, 'beta': s.beta, 'gamma': s.gamma})
#     except:
#         pass

# import pandas as pd
# r0_df = pd.DataFrame(results).sort_values('R0', ascending=False)
# print(r0_df.to_string(index=False))
import numpy as np
import pandas as pd
from src.models.sir_model import SIRModel

results = []

for country in featured['country'].unique()[:20]:
    sub = featured[featured.country == country].sort_values('date')
    I = sub['new_cases_7d'].fillna(0).values

    # Safe population handling
    if 'population' in sub and sub['population'].notna().any():
        pop_val = sub['population'].median()
        pop = int(pop_val) if not np.isnan(pop_val) else 1_000_000
    else:
        pop = 1_000_000

    # Skip countries with too little data
    if len(I) < 50 or I.max() < 10:
        continue

    try:
        s = SIRModel()
        s.fit(I, N=max(pop, 1_000_000))
        results.append({
            'country': country,
            'R0': s.r0,
            'beta': s.beta,
            'gamma': s.gamma
        })
    except Exception as e:
        print(f"Skipping {country} due to error: {e}")
        continue

# Collect results into DataFrame
r0_df = pd.DataFrame(results).sort_values('R0', ascending=False)
print(r0_df.to_string(index=False))


  SIR fit  β=0.3000  γ=0.1000  R0=3.00
  SIR fit  β=1.4178  γ=1.4345  R0=0.99
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.3757  γ=0.3717  R0=1.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.0102  γ=1.9998  R0=0.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.7683  γ=0.7475  R0=1.03
  SIR fit  β=0.3000  γ=0.1000  R0=3.00
  SIR fit  β=0.2847  γ=0.2810  R0=1.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.0109  γ=1.9990  R0=0.01
  SIR fit  β=0.0189  γ=1.9908  R0=0.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.3000  γ=0.1000  R0=3.00
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
  SIR fit  β=0.0100  γ=2.0000  R0=0.01
            country       R0     beta    gamma
        Afghanistan 3.000000 0.300000 0.100000
            Belgium 3.000000 0.300000 0.100000
          Australia 3.000000 0.300000 0.100000
            Armenia 1.027854 0.768296 0.747476
            Austria 1.01